<a href="https://colab.research.google.com/github/ajithosh229/MACHINE-LEARNING/blob/main/Week_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q pgmpy ucimlrepo pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 11.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

In [ ]:
heart = fetch_ucirepo(id=45)


In [ ]:
X = heart.data.features
y = heart.data.targets


In [ ]:
X

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,45,1,1,110,264,0,0,132,0,1.2,2,0.0,7.0
299,68,1,4,144,193,1,0,141,0,3.4,2,2.0,7.0
300,57,1,4,130,131,0,0,115,1,1.2,2,1.0,7.0
301,57,0,2,130,236,0,2,174,0,0.0,2,1.0,3.0


In [ ]:
y

,num
0,0
1,2
2,1
3,0
4,0
...,...
298,1
299,2
300,3
301,1


In [ ]:
data = pd.concat([X, y], axis=1)


In [ ]:
columns = ["age", "sex", "cp", "exang", "num"]
data = data[columns].copy()


In [ ]:
data

,age,sex,cp,exang,num
0,63,1,1,0,0
1,67,1,4,1,2
2,67,1,4,1,1
3,37,1,3,0,0
4,41,0,2,0,0
...,...,...,...,...,...
298,45,1,1,0,1
299,68,1,4,0,2
300,57,1,4,1,3
301,57,0,2,0,1


In [ ]:
for col in columns:
  data[col] = pd.to_numeric(data[col], errors="coerce")

In [ ]:
data = data.dropna(subset=["num"])
for col in ["age", "sex", "cp", "exang"]:
    data[col] = data[col].fillna(data[col].median())

In [ ]:
data

,age,sex,cp,exang,num
0,63,1,1,0,0
1,67,1,4,1,2
2,67,1,4,1,1
3,37,1,3,0,0
4,41,0,2,0,0
...,...,...,...,...,...
298,45,1,1,0,1
299,68,1,4,0,2
300,57,1,4,1,3
301,57,0,2,0,1


In [ ]:
data["age_group"] = pd.cut(
  data["age"],
  bins=[0, 40, 55, 100],
  labels=[0, 1, 2],
  include_lowest=True
  ).astype(int)



In [ ]:
data["HeartDisease"] = (data["num"] > 0).astype(int)

In [ ]:
data = data[
  ["age_group", "sex", "cp", "exang", "HeartDisease"]
  ].astype(int)



In [ ]:
model = DiscreteBayesianNetwork([
  ("age_group", "HeartDisease"),
  ("sex", "HeartDisease"),
  ("cp", "HeartDisease"),
  ("exang", "HeartDisease")
  ])



In [ ]:
estimator = BayesianEstimator(model, data)
cpds = estimator.get_parameters(prior_type="BDeu", equivalent_sample_size=10)
model.add_cpds(*cpds)


/tmp/ipykernel_2423/1448945110.py:1: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(model, data)


In [ ]:
print("Model valid:", model.check_model())

Model valid: True


In [ ]:
print("\nConditional Probability Distributions:")
for cpd in model.get_cpds():
    print(cpd)
    print()


Conditional Probability Distributions:
+--------------+-----------+
| age_group(0) | 0.0681576 |
+--------------+-----------+
| age_group(1) | 0.43557   |
+--------------+-----------+
| age_group(2) | 0.496273  |
+--------------+-----------+

+-----------------+--------------+-----+--------------------+
| age_group       | age_group(0) | ... | age_group(2)       |
+-----------------+--------------+-----+--------------------+
| cp              | cp(1)        | ... | cp(4)              |
+-----------------+--------------+-----+--------------------+
| exang           | exang(0)     | ... | exang(1)           |
+-----------------+--------------+-----+--------------------+
| sex             | sex(0)       | ... | sex(1)             |
+-----------------+--------------+-----+--------------------+
| HeartDisease(0) | 0.5          | ... | 0.0907429963459196 |
+-----------------+--------------+-----+--------------------+
| HeartDisease(1) | 0.5          | ... | 0.9092570036540804 |
+-----------

In [ ]:
inference = VariableElimination(model)

In [ ]:
patient = {
  "age_group": 2,
  "sex": 1,
  "cp": 4,
  "exang": 1
  }



In [ ]:
result = inference.query(
  variables=["HeartDisease"],
  evidence=patient,
  show_progress=False
  )



In [ ]:
states = result.state_names["HeartDisease"]
probabilities = result.values
print("Patient Details:", patient)
print("\nDiagnosis Probabilities:")

Patient Details: {'age_group': 2, 'sex': 1, 'cp': 4, 'exang': 1}

Diagnosis Probabilities:


In [ ]:
for state, probability in zip(states, probabilities):
  label = "Heart Disease Present" if state == 1 else "No Heart Disease"
  print(f"{label}: {probability * 100:.2f}%")


No Heart Disease: 9.07%
Heart Disease Present: 90.93%


In [ ]:
print("\nMost probable model outcome:",
"Heart Disease Present" if probabilities[1] > probabilities[0]
else "No Heart Disease")



Most probable model outcome: Heart Disease Present
